# gambit_arm: overview

Everything that moves the SO-ARM101: servo protocol, kinematics, board geometry, safety,
pick-and-place, grasp verification, plus the run notebooks that talk to the real arm. The
real-time path lives outside Python: `gambit_cpp/` (C++20 daemon, same protocol and FK) and
`firmware/stm32/` (bare-metal servo-bus owner). Limits: `gambit_bringup/config/safety.yaml`;
geometry: `gambit_bringup/config/board.yaml`.

**Deliverable checklist (README, "Hardware and execution")** mapped to notebooks:

| Milestone | Where |
|---|---|
| Arm assembled, servo IDs assigned, calibrated, teleoperating | `servo_bus` -> `run_servo_setup` -> `run_teleop` |
| Baseline repeatability (same pose 100x, spread in mm) | `kinematics` + `run_repeatability` |
| Board registered with fixed corner markers | `square_map` (uses board.yaml; marker detection is gambit_vision) |
| Rigid mounting of arm and board on one plate | hardware; re-measure board.yaml + `run_repeatability` afterwards |
| Gripper fingers shaped to pieces, with grasp verification | `grasp_verify` |
| Pick-and-place first attempt 55-60 % | `pick_place` + `run_pick_place_trials` |
| After verify-and-retry 95 % | `grasp_verify` + `run_pick_place_trials` |
| Per-move success >= 99 % (stretch) | `run_pick_place_trials` (full move sequences incl. captures) |
| Full game with no human intervention (10 games) | gambit_game + `pick_place.plan_chess_move` |
| Learned policy vs scripted | `run_train_policy` |

**TODO (package level)**
- [ ] finish the TODOs in each library notebook (each has its own checklist)
- [ ] hardware: IDs, calibration, teleop, repeatability numbers
- [ ] trials: first-attempt and after-retry rates
- [ ] Python vs C++ (and STM32) comparison, below

## Run order

Library notebooks (each `%run`s the ones above it; all run in seconds, no hardware):

1. `square_map.ipynb`: squares <-> world points, graveyard, grasp/hover heights
2. `servo_bus.ipynb`: STS3215 packets, ticks <-> radians, `ServoBus` (I/O TODO)
3. `kinematics.ipynb`: FK from the URDF, Jacobian, DLS IK (uses square_map)
4. `safety.ipynb`: limits, workspace, e-stop (uses kinematics)
5. `pick_place.ipynb`: primitives, chess-move sequences, planning, backends (uses safety)
6. `grasp_verify.ipynb`: verdict + verify-and-retry loop (uses pick_place)

Run notebooks (hardware / GPU; never executed by CI):

1. `run_servo_setup.ipynb`: assign IDs 1..6, calibrate offsets, write calibration
2. `run_teleop.ipynb`: LeRobot leader/follower teleoperation, record demonstrations
3. `run_repeatability.ipynb`: same pose 100x, spread in mm
4. `run_pick_place_trials.ipynb`: success rates vs the 55-60 % / 95 % targets
5. `run_train_policy.ipynb`: LeRobot imitation learning (Colab-ready), scripted vs learned

## Python vs C++ path: comparison plan

README: "get the arm working and trained on the laptop first, then add the STM32 firmware as
a real-time layer. Measure both paths on the same task." Paths:

| Path | Stack |
|---|---|
| A. Python | `pick_place` -> `ServoBackend` -> `ServoBus` (pyserial) -> adapter |
| B. C++ | planner -> IPC -> `gambit_arm_daemon` (C++, termios) -> adapter |
| C. C++ + STM32 | planner -> daemon -> host link (protocol.md) -> STM32 -> servo bus |

Same task for all: the 100-pose repeatability run and a fixed list of 50 pick-and-place moves.
Metrics: PING round-trip time (`ServoBus.round_trip_time`), control-loop period and jitter
(p50 / p99 / max of the time between SYNC_WRITEs), repeatability spread (mm), pick-and-place
first-attempt success, and time per move. Record in one table per path; the C++ unit tests
use the same packet bytes and FK numbers as the notebooks so the paths are comparable.

In [ ]:
from pathlib import Path
NOTEBOOKS = ["square_map", "servo_bus", "kinematics", "safety", "pick_place", "grasp_verify",
             "run_servo_setup", "run_teleop", "run_repeatability", "run_pick_place_trials",
             "run_train_policy"]

## Self-check

In [ ]:
missing = [n for n in NOTEBOOKS if not Path(f"{n}.ipynb").exists()]
assert not missing, missing
print("overview OK")